In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))

import logging
logging.basicConfig(level=logging.INFO, force=True)

from src.data.ingestion.statcast_client import fetch_days, project_root

DATES = [
    "2024-04-15",   # already cached
    "2024-06-15",
    "2024-08-15",
]

df = fetch_days(DATES)
print(df.shape)
print(df.groupby("game_date").size())

INFO:src.data.ingestion.statcast_client:reusing snapshot statcast_2024-04-15_ingested_2026-08-31.parquet
INFO:src.data.ingestion.statcast_client:downloading statcast for 2024-06-15


This is a large query, it may take a moment to complete


100%|██████████| 1/1 [00:06<00:00,  6.72s/it]
INFO:src.data.ingestion.statcast_client:wrote statcast_2024-06-15_ingested_2026-09-01.parquet (4145 rows, 0.78 MB)
INFO:src.data.ingestion.statcast_client:downloading statcast for 2024-08-15


This is a large query, it may take a moment to complete


100%|██████████| 1/1 [00:00<00:00,  9.57it/s]
INFO:src.data.ingestion.statcast_client:wrote statcast_2024-08-15_ingested_2026-09-01.parquet (2147 rows, 0.45 MB)


(10654, 119)
game_date
2024-04-15    4362
2024-06-15    4145
2024-08-15    2147
dtype: int64


In [2]:
raw_dir = project_root() / "data" / "raw"
for p in sorted(raw_dir.glob("*.parquet")):
    print(p.name, f"{p.stat().st_size/1e6:.2f} MB")

statcast_2024-04-15_ingested_2026-08-31.parquet 0.82 MB
statcast_2024-06-15_ingested_2026-09-01.parquet 0.78 MB
statcast_2024-08-15_ingested_2026-09-01.parquet 0.45 MB


In [3]:
import duckdb

con = duckdb.connect()

glob_path = str(project_root() / "data" / "raw" / "statcast_*.parquet")

con.execute(f"""
    CREATE OR REPLACE VIEW pitches AS
    SELECT * FROM read_parquet('{glob_path}')
""")

print(con.execute("SELECT COUNT(*) FROM pitches").fetchone())

(10654,)


In [4]:
q = """
SELECT
    game_date,
    COUNT(*)                                   AS pitches,
    COUNT(DISTINCT game_pk)                    AS games,
    COUNT(DISTINCT (game_pk, at_bat_number))   AS plate_appearances,
    ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT (game_pk, at_bat_number)), 2) AS pitches_per_pa
FROM pitches
GROUP BY game_date
ORDER BY game_date
"""
con.execute(q).df()

,game_date,pitches,games,plate_appearances,pitches_per_pa
0,2024-04-15,4362,15,1111,3.93
1,2024-06-15,4145,14,1051,3.94
2,2024-08-15,2147,7,522,4.11


In [5]:
q = """
SELECT
    pitch_type,
    COUNT(*)                                        AS pitches,
    SUM(CASE WHEN description IN (
        'foul','hit_into_play','swinging_strike',
        'swinging_strike_blocked','foul_tip') THEN 1 ELSE 0 END) AS swings,
    SUM(CASE WHEN description IN (
        'swinging_strike','swinging_strike_blocked') THEN 1 ELSE 0 END) AS whiffs,
    ROUND(AVG(release_speed), 1)                    AS avg_velo
FROM pitches
WHERE pitch_type IS NOT NULL
GROUP BY pitch_type
HAVING swings >= 100
ORDER BY whiffs * 1.0 / swings DESC
"""
con.execute(q).df()

,pitch_type,pitches,swings,whiffs,avg_velo
0,FS,385,201.0,68.0,86.4
1,SL,1407,673.0,212.0,86.0
2,CH,1084,541.0,163.0,85.5
3,ST,666,302.0,81.0,82.1
4,CU,765,332.0,77.0,79.5
5,FC,909,436.0,82.0,89.0
6,FF,3541,1700.0,298.0,94.1
7,SI,1605,742.0,74.0,93.0


In [6]:
import pandas as pd
from src.features.plate_discipline import add_swing_flags

pdf = pd.read_parquet(list(raw_dir.glob("statcast_*.parquet")))
pdf = add_swing_flags(pdf)

python_side = (
    pdf[pdf["pitch_type"].notna()]
    .groupby("pitch_type")
    .agg(swings=("is_swing", "sum"), whiffs=("is_whiff", "sum"))
)
python_side = python_side[python_side["swings"] >= 100]
python_side["whiff_pct"] = (python_side["whiffs"] / python_side["swings"]).round(3)
print(python_side.sort_values("whiff_pct", ascending=False).to_string())

            swings  whiffs  whiff_pct
pitch_type                           
FS             201      68      0.338
SL             673     212      0.315
CH             541     163      0.301
ST             302      81      0.268
CU             332      77      0.232
FC             436      82      0.188
FF            1700     298      0.175
SI             742      74      0.100
